# 3 · Evaluación de la Parte 1: comparativa y experimentos

Práctica final · **Asistente de agenda en lenguaje natural** · Procesamiento del Lenguaje Natural

Poned en la misma carpeta este notebook, `utilidades.py`, `evaluar.py`, `train.jsonl` y `agenda.jsonl`. Las celdas marcadas con `# TODO` son las que tenéis que completar.

Este notebook lee `resultados_A.json`, `resultados_B.json` y `resultados_C.json` (generados por los notebooks 2A, 2B y 2C).

In [ ]:
import json, os
import pandas as pd
import matplotlib.pyplot as plt
from utilidades import cargar, guardar, particiones
from evaluar import evaluar, imprimir

train, val = particiones()
RES = {t: json.load(open(f"resultados_{t}.json", encoding="utf-8")) for t in "ABC" if os.path.exists(f"resultados_{t}.json")}
print("Resultados disponibles:", list(RES))

## Tabla comparativa
Completad las columnas que faltan (datos de entrenamiento, tamaño del modelo, ¿interpretable?).

In [ ]:
filas = []
for t, res in RES.items():
    for nombre, r in res.items():
        filas.append({"tarea": t, "enfoque": nombre, "métrica (validación)": r["metrica"], "frases difíciles": r["dificiles"],
                      "ms por frase": r["ms_por_frase"], "datos de entrenamiento": "", "tamaño del modelo": "", "¿interpretable?": ""})
tabla = pd.DataFrame(filas)
tabla.round(3)

## E1 · Curva de aprendizaje (tarea A)
A2, A3, A6 y A7 con 25, 50, 100 y todas las frases, al menos 3 semillas; el LLM sin entrenar (A8) como línea horizontal. Necesita `tarea_a.py` (ver notebook 2A).

In [ ]:
import random
from collections import defaultdict
from sklearn.metrics import f1_score

def submuestra(n, semilla):
    """n frases estratificadas por intención."""
    rnd = random.Random(semilla); por = defaultdict(list)
    for x in train: por[x["intencion"]].append(x)
    return [y for xs in por.values() for y in rnd.sample(xs, min(len(xs), max(1, round(n * len(xs) / len(train)))))]

try:
    from tarea_a import entrenar_y_predecir
    curva = defaultdict(dict)
    for enfoque in ["A2", "A3", "A6", "A7"]:
        for n in [25, 50, 100, len(train)]:
            f1s = []
            for s in range(3):
                sub = submuestra(n, s)
                pred = entrenar_y_predecir(enfoque, [x["texto"] for x in sub], [x["intencion"] for x in sub], [x["texto"] for x in val])
                f1s.append(f1_score([x["intencion"] for x in val], pred, average="macro"))
            curva[enfoque][n] = sum(f1s) / len(f1s)
    # TODO: dibujar la curva y la línea horizontal del LLM
except ImportError:
    print("Falta tarea_a.py con la función entrenar_y_predecir (ver notebook 2A)")

## E2 · Ablación del preprocesado
A2, A3 y A6 con texto original, normalizado, sin stopwords y con stemming (usad `normalizar` de `utilidades.py`).

In [ ]:
# TODO: tabla preprocesado × enfoque con la F1 macro en validación

## E3 · Robustez
Cada enfoque en frases normales y difíciles. Construid un subconjunto difícil de al menos 30 frases en validación.

In [ ]:
print("Frases difíciles en validación:", sum(x["dificil"] for x in val))
# La columna «frases difíciles» de la tabla comparativa ya la calcula evaluar(val, preds, "dificil")
# TODO: comparar normales frente a difíciles y comentarlo

## E4 · Análisis de errores
Al menos 10 errores por familia (reglas, clásicos, embeddings, Transformers, LLM), agrupados por causa, y una mejora para la causa más frecuente.

In [ ]:
def errores_A(nombre, n=10):
    P = {p["id"]: p["intencion"] for p in RES["A"][nombre]["preds"]}
    return pd.DataFrame([(x["texto"], x["intencion"], P[x["id"]]) for x in val if P.get(x["id"]) != x["intencion"]][:n],
                        columns=["frase", "oro", "predicción"])
if RES.get("A"):
    print(errores_A(next(iter(RES["A"]))))
# TODO: lo mismo para B y C, y agrupar por causa

## Función para el test oculto (se entrega)
El profesor ejecutará `generar_predicciones` con el **test oculto**, que no veréis: un fichero con el mismo formato que `train.jsonl` pero sin etiquetas. Usa los enfoques elegidos que habéis exportado a `tarea_a.py`, `tarea_b.py` y `tarea_c.py`. Comprobad aquí que funciona con vuestra validación.

In [ ]:
def generar_predicciones(ruta_textos, ruta_salida):
    from tarea_a import predecir_intencion
    from tarea_b import extraer_entidades
    from tarea_c import ranking_agenda
    agenda = cargar("agenda.jsonl")
    preds = [{"id": x["id"],
              "intencion": predecir_intencion(x["texto"]),
              "entidades": extraer_entidades(x["texto"]),
              "ref_ranking": ranking_agenda(x["texto"], agenda)}
             for x in cargar(ruta_textos)]
    guardar(preds, ruta_salida)
    return preds

# Comprobación con vuestra validación: el profesor hará lo mismo con el test oculto
try:
    pred = generar_predicciones("mi_validacion.jsonl", "pred_validacion.jsonl")
    imprimir(evaluar(val, pred), "Sistema completo · validación")
except ImportError as e:
    print("Faltan tarea_a.py, tarea_b.py o tarea_c.py (ver los notebooks 2A, 2B y 2C):", e)